# Normalizzazione `df_final_FEB_2026`

Questo notebook parte dal file finale FEB e costruisce una tabella canonica in formato long, pensata come esempio pratico di una prima Delta table appendabile.

In [1]:
import hashlib
import os
import sys
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent
input_path = project_root / "data" / "output" / "df_final_FEB_2026.xlsx"
output_path = project_root / "data" / "output" / "df_canonical_FEB_2026.csv"

print("python:", sys.executable)
print("pid:", os.getpid())
print("project_root:", project_root)
print("input_path:", input_path)
print("output_path:", output_path)
print("file_exists:", input_path.exists())

python: c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sys_perlettura_PDF\PDFDataExtractor-pipeline\.venv\Scripts\python.exe
pid: 27036
project_root: c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sys_perlettura_PDF\PDFDataExtractor-pipeline
input_path: c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sys_perlettura_PDF\PDFDataExtractor-pipeline\data\output\df_final_FEB_2026.xlsx
output_path: c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sys_perlettura_PDF\PDFDataExtractor-pipeline\data\output\df_canonical_FEB_2026.csv
file_exists: True


In [2]:
def _excel_serial_to_datetime(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    return pd.Timestamp("1899-12-30") + pd.to_timedelta(numeric, unit="D")


def _read_xlsx_fallback(path: Path) -> pd.DataFrame:
    ns = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}

    with zipfile.ZipFile(path) as zf:
        shared_strings = []
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for si in root.findall("m:si", ns):
                texts = [t.text or "" for t in si.iterfind(".//m:t", ns)]
                shared_strings.append("".join(texts))

        sheet = ET.fromstring(zf.read("xl/worksheets/sheet1.xml"))
        rows = []
        for row in sheet.findall(".//m:sheetData/m:row", ns):
            values = {}
            for cell in row.findall("m:c", ns):
                ref = cell.attrib.get("r", "")
                col = "".join(ch for ch in ref if ch.isalpha())
                cell_type = cell.attrib.get("t")
                value_node = cell.find("m:v", ns)
                value = ""

                if cell_type == "s" and value_node is not None and value_node.text is not None:
                    idx = int(value_node.text)
                    value = shared_strings[idx] if idx < len(shared_strings) else ""
                elif cell_type == "inlineStr":
                    value = "".join(t.text or "" for t in cell.iterfind(".//m:t", ns))
                elif value_node is not None and value_node.text is not None:
                    value = value_node.text

                values[col] = value
            rows.append(values)

    header_row = rows[0]
    col_order = sorted(header_row.keys(), key=lambda x: (len(x), x))
    headers = [header_row[c] for c in col_order]
    data = [[row.get(c, "") for c in col_order] for row in rows[1:]]
    return pd.DataFrame(data, columns=headers)


def read_final_dataset(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_excel(path)
        loader = "pandas.read_excel"
    except ImportError:
        df = _read_xlsx_fallback(path)
        loader = "xml_fallback"

    for col in [
        "page_num", "table_id", "2025_act", "2026_act", "2026_bdg",
        "eur_vs_act", "pct_vs_act", "eur_vs_bdg", "pct_vs_bdg", "eur_delta", "pct_delta"
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "period_start" in df.columns:
        if pd.api.types.is_numeric_dtype(df["period_start"]) or df["period_start"].astype(str).str.fullmatch(r"\d+(?:\.\d+)?").fillna(False).all():
            df["period_start"] = _excel_serial_to_datetime(df["period_start"])
        else:
            df["period_start"] = pd.to_datetime(df["period_start"], errors="coerce")

    if "timestamp_utc" in df.columns:
        if pd.api.types.is_numeric_dtype(df["timestamp_utc"]) or df["timestamp_utc"].astype(str).str.fullmatch(r"\d+(?:\.\d+)?").fillna(False).all():
            df["timestamp_utc"] = _excel_serial_to_datetime(df["timestamp_utc"])
        else:
            df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], errors="coerce")

    return df, loader

In [3]:
df_wide, loader = read_final_dataset(input_path)
print("loader:", loader)
print("shape:", df_wide.shape)
print(df_wide.dtypes.to_string())
df_wide.head()

loader: xml_fallback
shape: (1902, 23)
page_num                    int64
capitolo                   object
sottocapitolo              object
page_title                 object
page_subtitle              object
page_subtitle_1            object
table_id                    int64
period_desc                object
period_start       datetime64[ns]
item_agg_2                 object
item_agg_1                 object
item_agg                   object
2025_act                  float64
2026_act                  float64
2026_bdg                  float64
eur_vs_act                float64
pct_vs_act                float64
eur_vs_bdg                float64
pct_vs_bdg                float64
eur_delta                 float64
pct_delta                 float64
run_id                     object
timestamp_utc      datetime64[ns]


,page_num,capitolo,sottocapitolo,page_title,page_subtitle,page_subtitle_1,table_id,period_desc,period_start,item_agg_2,...,2026_act,2026_bdg,eur_vs_act,pct_vs_act,eur_vs_bdg,pct_vs_bdg,eur_delta,pct_delta,run_id,timestamp_utc
0,2,OVERVIEW,OVERVIEW,MENARINI GROUP PERFORMANCES OVERVIEW,,,0,MONTH,2026-02-01,,...,NaN,NaN,23054.0,0.068,11037.0,0.031,NaN,NaN,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630424
1,2,OVERVIEW,OVERVIEW,MENARINI GROUP PERFORMANCES OVERVIEW,,,0,MONTH,2026-02-01,,...,NaN,NaN,8965.0,0.061,-1391.0,-0.009,NaN,NaN,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630424
2,2,OVERVIEW,OVERVIEW,MENARINI GROUP PERFORMANCES OVERVIEW,,,0,MONTH,2026-02-01,,...,NaN,NaN,6377.0,0.059,4487.0,0.041,NaN,NaN,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630424
3,2,OVERVIEW,OVERVIEW,MENARINI GROUP PERFORMANCES OVERVIEW,,,0,MONTH,2026-02-01,,...,NaN,NaN,5301.0,0.301,4968.0,0.277,NaN,NaN,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630424
4,2,OVERVIEW,OVERVIEW,MENARINI GROUP PERFORMANCES OVERVIEW,,,0,MONTH,2026-02-01,,...,NaN,NaN,2736.0,0.065,464.0,0.010,NaN,NaN,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630424


## Mappatura verso il modello canonico

Le colonne misura wide vengono trasformate in righe tramite `metric_name`, `metric_unit`, `metric_value`, `reference_year`, `scenario`, `comparison_target`.

In [4]:
measure_mapping = {
    "2025_act": {"metric_name": "net_sales", "metric_unit": "eur_k", "reference_year": 2025, "scenario": "act", "comparison_target": None},
    "2026_act": {"metric_name": "net_sales", "metric_unit": "eur_k", "reference_year": 2026, "scenario": "act", "comparison_target": None},
    "2026_bdg": {"metric_name": "net_sales", "metric_unit": "eur_k", "reference_year": 2026, "scenario": "bdg", "comparison_target": None},
    "eur_vs_act": {"metric_name": "variance", "metric_unit": "eur_k", "reference_year": 2026, "scenario": "act", "comparison_target": "previous_year_act"},
    "pct_vs_act": {"metric_name": "variance", "metric_unit": "pct", "reference_year": 2026, "scenario": "act", "comparison_target": "previous_year_act"},
    "eur_vs_bdg": {"metric_name": "variance", "metric_unit": "eur_k", "reference_year": 2026, "scenario": "bdg", "comparison_target": "budget"},
    "pct_vs_bdg": {"metric_name": "variance", "metric_unit": "pct", "reference_year": 2026, "scenario": "bdg", "comparison_target": "budget"},
    "eur_delta": {"metric_name": "delta", "metric_unit": "eur_k", "reference_year": 2026, "scenario": "derived", "comparison_target": None},
    "pct_delta": {"metric_name": "delta", "metric_unit": "pct", "reference_year": 2026, "scenario": "derived", "comparison_target": None},
}

dimension_columns = [
    "page_num",
    "capitolo",
    "sottocapitolo",
    "page_title",
    "page_subtitle",
    "page_subtitle_1",
    "table_id",
    "period_desc",
    "period_start",
    "item_agg_2",
    "item_agg_1",
    "item_agg",
    "run_id",
    "timestamp_utc",
]

In [5]:
df_long = df_wide.melt(
    id_vars=dimension_columns,
    value_vars=list(measure_mapping.keys()),
    var_name="source_measure_column",
    value_name="metric_value",
)

mapping_df = pd.DataFrame.from_dict(measure_mapping, orient="index").reset_index(names="source_measure_column")
df_long = df_long.merge(mapping_df, on="source_measure_column", how="left")
df_long = df_long[df_long["metric_value"].notna()].copy()

df_long = df_long.rename(columns={
    "capitolo": "chapter_name",
    "sottocapitolo": "subchapter_name",
    "page_subtitle_1": "page_subtitle_detail",
    "period_desc": "period_scope",
    "period_start": "period_start_date",
    "item_agg": "entity_name",
    "item_agg_1": "entity_group_1",
    "item_agg_2": "entity_group_2",
})

df_long["report_year"] = 2026
df_long["report_month"] = 2
df_long["report_month_name"] = "FEBRUARY"
df_long["report_period_label"] = "FEB_2026"
df_long["source_file_name"] = input_path.name
df_long["source_file_path"] = str(input_path)
df_long["source_file_hash"] = hashlib.sha256(input_path.read_bytes()).hexdigest()
df_long["pipeline_version"] = "demo_from_final_file"
df_long["config_version"] = "demo_from_final_file"
df_long["load_date"] = pd.Timestamp.utcnow().normalize()

key_columns = [
    "source_file_hash",
    "page_num",
    "table_id",
    "period_scope",
    "entity_name",
    "entity_group_1",
    "entity_group_2",
    "metric_name",
    "metric_unit",
    "reference_year",
    "scenario",
    "comparison_target",
]
df_long["metric_row_id"] = (
    df_long[key_columns]
    .fillna("<NULL>")
    .astype(str)
    .agg("|".join, axis=1)
    .map(lambda s: hashlib.sha256(s.encode("utf-8")).hexdigest())
)

canonical_columns = [
    "metric_row_id",
    "run_id",
    "ingested_at_utc",
    "load_date",
    "source_file_name",
    "source_file_path",
    "source_file_hash",
    "pipeline_version",
    "config_version",
    "report_year",
    "report_month",
    "report_month_name",
    "report_period_label",
    "page_num",
    "page_title",
    "page_subtitle",
    "page_subtitle_detail",
    "table_id",
    "chapter_name",
    "subchapter_name",
    "period_scope",
    "period_start_date",
    "entity_name",
    "entity_group_1",
    "entity_group_2",
    "metric_name",
    "metric_unit",
    "metric_value",
    "reference_year",
    "scenario",
    "comparison_target",
    "source_measure_column",
]

df_long = df_long.rename(columns={"timestamp_utc": "ingested_at_utc"})
df_canonical = df_long[canonical_columns].copy()
df_canonical = df_canonical.sort_values([
    "page_num", "table_id", "period_scope", "entity_name", "entity_group_1", "entity_group_2", "metric_name", "metric_unit"
]).reset_index(drop=True)

print("wide shape:", df_wide.shape)
print("canonical shape:", df_canonical.shape)
df_canonical.head(20)

wide shape: (1902, 23)
canonical shape: (11530, 32)


,metric_row_id,run_id,ingested_at_utc,load_date,source_file_name,source_file_path,source_file_hash,pipeline_version,config_version,report_year,...,entity_name,entity_group_1,entity_group_2,metric_name,metric_unit,metric_value,reference_year,scenario,comparison_target,source_measure_column
0,866743ac6316c2793af7dca86af7b1ee50e45f4099bf40...,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630424,2026-03-30 00:00:00+00:00,df_final_FEB_2026.xlsx,c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sy...,ae49878602fe0e3b314b4f795c829d1a2abf078449ebe7...,demo_from_final_file,demo_from_final_file,2026,...,ASIA,Oncology,,variance,eur_k,100.000,2026,act,previous_year_act,eur_vs_act
1,eb09bdb34ecb537b2c77f8db63affff5e57046f9762f3b...,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630424,2026-03-30 00:00:00+00:00,df_final_FEB_2026.xlsx,c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sy...,ae49878602fe0e3b314b4f795c829d1a2abf078449ebe7...,demo_from_final_file,demo_from_final_file,2026,...,ASIA,Oncology,,variance,eur_k,-68.000,2026,bdg,budget,eur_vs_bdg
2,9d329f5786469dcd8325b3102101c46184d7526d964fcd...,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630424,2026-03-30 00:00:00+00:00,df_final_FEB_2026.xlsx,c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sy...,ae49878602fe0e3b314b4f795c829d1a2abf078449ebe7...,demo_from_final_file,demo_from_final_file,2026,...,ASIA,Oncology,,variance,pct,1.404,2026,act,previous_year_act,pct_vs_act
3,29dd44f33e5c612bbad7360e95c34bd4a9372b67a36254...,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630424,2026-03-30 00:00:00+00:00,df_final_FEB_2026.xlsx,c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sy...,ae49878602fe0e3b314b4f795c829d1a2abf078449ebe7...,demo_from_final_file,demo_from_final_file,2026,...,ASIA,Oncology,,variance,pct,-0.284,2026,bdg,budget,pct_vs_bdg
4,617514a4d23753bb07fbd8dbbec442f0ba8ca78356cbfc...,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630424,2026-03-30 00:00:00+00:00,df_final_FEB_2026.xlsx,c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sy...,ae49878602fe0e3b314b4f795c829d1a2abf078449ebe7...,demo_from_final_file,demo_from_final_file,2026,...,Asia Pacific,Pharma,,variance,eur_k,2736.000,2026,act,previous_year_act,eur_vs_act
5,cbc0f534fecf1fdf3ec8efdb02949d357fd551c4c00e13...,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630424,2026-03-30 00:00:00+00:00,df_final_FEB_2026.xlsx,c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sy...,ae49878602fe0e3b314b4f795c829d1a2abf078449ebe7...,demo_from_final_file,demo_from_final_file,2026,...,Asia Pacific,Pharma,,variance,eur_k,464.000,2026,bdg,budget,eur_vs_bdg
6,cf16a701fd95f0f071428195231ae75fc5360866590f37...,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630424,2026-03-30 00:00:00+00:00,df_final_FEB_2026.xlsx,c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sy...,ae49878602fe0e3b314b4f795c829d1a2abf078449ebe7...,demo_from_final_file,demo_from_final_file,2026,...,Asia Pacific,Pharma,,variance,pct,0.065,2026,act,previous_year_act,pct_vs_act
7,3215b61c8b200fd28a047391215c9409f8a79f5a364c4a...,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630424,2026-03-30 00:00:00+00:00,df_final_FEB_2026.xlsx,c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sy...,ae49878602fe0e3b314b4f795c829d1a2abf078449ebe7...,demo_from_final_file,demo_from_final_file,2026,...,Asia Pacific,Pharma,,variance,pct,0.010,2026,bdg,budget,pct_vs_bdg
8,000f1756b771f0b1f6d87e9ad12a645d3a9442ebff46ca...,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630424,2026-03-30 00:00:00+00:00,df_final_FEB_2026.xlsx,c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sy...,ae49878602fe0e3b314b4f795c829d1a2abf078449ebe7...,demo_from_final_file,demo_from_final_file,2026,...,Cell Biology,Other Business,,variance,eur_k,83.000,2026,act,previous_year_act,eur_vs_act
9,1ac48f32e28892dce5dc986b5b08c2f13d174f0fbc4c48...,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630424,2026-03-30 00:00:00+00:00,df_final_FEB_2026.xlsx,c:\work\

In [6]:
pd.DataFrame({
    "column": df_canonical.columns,
    "dtype": [str(df_canonical[c].dtype) for c in df_canonical.columns],
})

,column,dtype
0,metric_row_id,object
1,run_id,object
2,ingested_at_utc,datetime64[ns]
3,load_date,"datetime64[us, UTC]"
4,source_file_name,object
5,source_file_path,object
6,source_file_hash,object
7,pipeline_version,object
8,config_version,object
9,report_year,int64


In [7]:
df_canonical[[
    "report_period_label",
    "page_num",
    "table_id",
    "chapter_name",
    "subchapter_name",
    "period_scope",
    "entity_name",
    "entity_group_1",
    "entity_group_2",
    "metric_name",
    "metric_unit",
    "metric_value",
    "reference_year",
    "scenario",
    "comparison_target",
    "source_measure_column",
]].head(30)

,report_period_label,page_num,table_id,chapter_name,subchapter_name,period_scope,entity_name,entity_group_1,entity_group_2,metric_name,metric_unit,metric_value,reference_year,scenario,comparison_target,source_measure_column
0,FEB_2026,2,0,OVERVIEW,OVERVIEW,MONTH,ASIA,Oncology,,variance,eur_k,100.000,2026,act,previous_year_act,eur_vs_act
1,FEB_2026,2,0,OVERVIEW,OVERVIEW,MONTH,ASIA,Oncology,,variance,eur_k,-68.000,2026,bdg,budget,eur_vs_bdg
2,FEB_2026,2,0,OVERVIEW,OVERVIEW,MONTH,ASIA,Oncology,,variance,pct,1.404,2026,act,previous_year_act,pct_vs_act
3,FEB_2026,2,0,OVERVIEW,OVERVIEW,MONTH,ASIA,Oncology,,variance,pct,-0.284,2026,bdg,budget,pct_vs_bdg
4,FEB_2026,2,0,OVERVIEW,OVERVIEW,MONTH,Asia Pacific,Pharma,,variance,eur_k,2736.000,2026,act,previous_year_act,eur_vs_act
5,FEB_2026,2,0,OVERVIEW,OVERVIEW,MONTH,Asia Pacific,Pharma,,variance,eur_k,464.000,2026,bdg,budget,eur_vs_bdg
6,FEB_2026,2,0,OVERVIEW,OVERVIEW,MONTH,Asia Pacific,Pharma,,variance,pct,0.065,2026,act,previous_year_act,pct_vs_act
7,FEB_2026,2,0,OVERVIEW,OVERVIEW,MONTH,Asia Pacific,Pharma,,variance,pct,0.010,2026,bdg,budget,pct_vs_bdg
8,FEB_2026,2,0,OVERVIEW,OVERVIEW,MONTH,Cell Biology,Other Business,,variance,eur_k,83.000,2026,act,previous_year_act,eur_vs_act
9,FEB_2026,2,0,OVERVIEW,OVERVIEW,MONTH,Cell Biology,Other Business,,variance,eur_k,-484.000,2026,bdg,budget,eur_vs_bdg


In [8]:
output_path.parent.mkdir(parents=True, exist_ok=True)
df_canonical.to_csv(output_path, index=False)
print(output_path)
print("saved_rows:", len(df_canonical))

c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sys_perlettura_PDF\PDFDataExtractor-pipeline\data\output\df_canonical_FEB_2026.csv
saved_rows: 11530
